In [1]:
%pip install yfinance tensorflow

import yfinance as yf
import pandas as pd
import numpy as np
import random
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

tickers = ["QQQ", "BB", "M", "CMG"]

SENTIMENT_FILES = {
    "QQQ": "../Sentiment/QQQ_Sentiment.csv",
    "BB": "../Sentiment/BB_Sentiment.csv",
    "M": "../Sentiment/M_Sentiment.csv",
    "CMG": "../Sentiment/CMG_Sentiment.csv"
}

In [3]:
def merge_sentiment(price, sentiment_path):
    sentiment_df = pd.read_csv(sentiment_path)

    sentiment_df["date"] = pd.to_datetime(sentiment_df["date"], errors="coerce")
    sentiment_df["sentiment"] = pd.to_numeric(sentiment_df["sentiment"], errors="coerce")
    sentiment_df = sentiment_df.dropna(subset=["date", "sentiment"])

    if getattr(sentiment_df["date"].dt, "tz", None) is not None:
        sentiment_df["date"] = sentiment_df["date"].dt.tz_localize(None)

    sentiment_df["date"] = sentiment_df["date"].dt.normalize()
    sentiment_df = sentiment_df.groupby("date", as_index=False)["sentiment"].mean()

    price = price.copy()
    price.index = pd.to_datetime(price.index)

    if getattr(price.index, "tz", None) is not None:
        price.index = price.index.tz_localize(None)

    price.index = price.index.normalize()

    price_reset = price.reset_index()
    price_reset = price_reset.rename(columns={price_reset.columns[0]: "date"})
    price_reset["date"] = pd.to_datetime(price_reset["date"], errors="coerce").dt.normalize()

    merged = price_reset.merge(sentiment_df, on="date", how="left")
    merged["sentiment"] = merged["sentiment"].fillna(0)
    merged = merged.set_index("date")

    return merged

In [4]:
prices = {}

for ticker_symbol in tickers:
    ticker = yf.Ticker(ticker_symbol)
    price = ticker.history(start="2009-02-14", end="2020-06-11")

    price = merge_sentiment(price, SENTIMENT_FILES[ticker_symbol])

    prices[ticker_symbol] = price

    print(f"\n{ticker_symbol}:")
    print(price[["Open", "Close", "sentiment"]].head())


QQQ:
                 Open      Close  sentiment
date                                       
2009-02-17  25.455397  25.222336        0.0
2009-02-18  25.386339  25.239597        0.0
2009-02-19  25.394969  24.851162        0.0
2009-02-20  24.574956  24.920233        0.0
2009-02-23  25.058336  24.048407        0.0

BB:
                 Open      Close  sentiment
date                                       
2009-02-17  46.540001  44.639999        0.0
2009-02-18  45.500000  42.099998        0.0
2009-02-19  42.419998  42.090000        0.0
2009-02-20  40.610001  39.150002        0.0
2009-02-23  39.840000  37.419998        0.0

M:
                Open     Close  sentiment
date                                     
2009-02-17  4.532084  4.543329        0.0
2009-02-18  4.593937  4.436495        0.0
2009-02-19  4.560198  4.307166        0.0
2009-02-20  4.245313  4.419624        0.0
2009-02-23  4.498346  4.160970        0.0

CMG:
              Open   Close  sentiment
date                           

In [5]:
for ticker_symbol in tickers:
    price = prices[ticker_symbol]

    price["Score"] = (price["Close"].shift(-1) > price["Close"]).astype(int)

    price["Change"]      = price["Close"].shift(1) - price["Open"].shift(1)
    price["%Change"]     = (price["Close"].shift(1) - price["Open"].shift(1)) / price["Open"].shift(1)
    price["CloseToOpen"] = price["Open"].shift(1) - price["Close"].shift(1)
    price["YestScore"]   = price["Score"].shift(1)
    price["5DayMean"]    = price["Close"].rolling(5).mean().shift(1)
    price["5DayVoli"]    = price["Close"].rolling(5).std().shift(1)
    price["5DayPerf"]    = price["Score"].rolling(5).sum().shift(1)

    price["sentiment"] = pd.to_numeric(price["sentiment"], errors="coerce").fillna(0)

    price.dropna(inplace=True)
    prices[ticker_symbol] = price

In [6]:
technical_features = [
    "Change", "%Change", "CloseToOpen", "YestScore",
    "5DayMean", "5DayVoli", "5DayPerf"
]

sentiment_features = ["sentiment"]

window_size = 5
all_results = {}

def make_late_fusion_windows(df, technical_features, sentiment_features, target_col, window_size):
    X_tech = []
    X_sent = []
    y = []
    d = []

    for i in range(window_size, len(df)):
        X_tech.append(df[technical_features].iloc[i-window_size:i].values)
        X_sent.append(df[sentiment_features].iloc[i-window_size:i].values)
        y.append(df[target_col].iloc[i])
        d.append(df.index[i])

    return np.array(X_tech), np.array(X_sent), np.array(y), np.array(d)

In [7]:
def build_late_fusion_model(window_size, n_tech_features, n_sent_features):
    tech_input = Input(shape=(window_size, n_tech_features), name="tech_input")
    tech_branch = LSTM(32, name="tech_lstm")(tech_input)
    tech_branch = Dropout(0.2)(tech_branch)

    sent_input = Input(shape=(window_size, n_sent_features), name="sent_input")
    sent_branch = LSTM(8, name="sent_lstm")(sent_input)
    sent_branch = Dropout(0.2)(sent_branch)

    merged = Concatenate()([tech_branch, sent_branch])
    output = Dense(1, activation="sigmoid")(merged)

    model = Model(inputs=[tech_input, sent_input], outputs=output)

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [8]:
for ticker_symbol in tickers:
    price = prices[ticker_symbol]

    priceTrain = price[price.index <= "2018-06-11"].copy()
    priceTest  = price[price.index > "2018-06-11"].copy()

    tech_scaler = StandardScaler()
    sent_scaler = StandardScaler()

    priceTrain_scaled = priceTrain.copy()
    priceTest_scaled = priceTest.copy()

    priceTrain_scaled[technical_features] = tech_scaler.fit_transform(priceTrain[technical_features])
    priceTest_scaled[technical_features] = tech_scaler.transform(priceTest[technical_features])

    priceTrain_scaled[sentiment_features] = sent_scaler.fit_transform(priceTrain[sentiment_features])
    priceTest_scaled[sentiment_features] = sent_scaler.transform(priceTest[sentiment_features])

    X_train_tech, X_train_sent, y_train, d_train = make_late_fusion_windows(
        priceTrain_scaled, technical_features, sentiment_features, "Score", window_size
    )

    combined_test = pd.concat([priceTrain_scaled.tail(window_size), priceTest_scaled])

    X_test_tech, X_test_sent, y_test, d_test = make_late_fusion_windows(
        combined_test, technical_features, sentiment_features, "Score", window_size
    )

    # keep only rows that belong to actual test dates
    test_mask = d_test > pd.Timestamp("2018-06-11")
    X_test_tech = X_test_tech[test_mask]
    X_test_sent = X_test_sent[test_mask]
    y_test = y_test[test_mask]
    d_test = d_test[test_mask]

    model = build_late_fusion_model(
        window_size,
        len(technical_features),
        len(sentiment_features)
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        [X_train_tech, X_train_sent],
        y_train,
        validation_split=0.2,
        epochs=20,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    probs = model.predict([X_test_tech, X_test_sent], verbose=0).flatten()
    preds = (probs >= 0.5).astype(int)

    results = pd.DataFrame({
        "Date": d_test,
        "Score": y_test,
        "Prediction": preds,
        "Probability": probs
    })

    all_results[ticker_symbol] = results

    print(f"\nFinished {ticker_symbol}")
    print(results.head())


Finished QQQ
        Date  Score  Prediction  Probability
0 2018-06-12      0           1     0.503954
1 2018-06-13      1           0     0.479327
2 2018-06-14      0           0     0.432348
3 2018-06-15      0           0     0.432610
4 2018-06-18      0           0     0.414481

Finished BB
        Date  Score  Prediction  Probability
0 2018-06-12      0           0     0.420778
1 2018-06-13      0           0     0.443940
2 2018-06-14      1           0     0.466854
3 2018-06-15      0           0     0.477542
4 2018-06-18      0           0     0.477797

Finished M
        Date  Score  Prediction  Probability
0 2018-06-12      0           1     0.528113
1 2018-06-13      0           1     0.507946
2 2018-06-14      1           1     0.567155
3 2018-06-15      1           1     0.581732
4 2018-06-18      1           1     0.575664

Finished CMG
        Date  Score  Prediction  Probability
0 2018-06-12      0           0     0.489965
1 2018-06-13      1           1     0.527733
2 

In [9]:
for ticker_symbol, results in all_results.items():
    score = results["Score"]
    pred = results["Prediction"]

    accuracy  = accuracy_score(score, pred)
    precision = precision_score(score, pred, zero_division=0)
    recall    = recall_score(score, pred, zero_division=0)
    f1        = f1_score(score, pred, zero_division=0)
    mcc       = matthews_corrcoef(score, pred)

    print(f"\n{ticker_symbol}:")
    print("Mean:     ", results["Score"].mean())
    print("Accuracy: ", accuracy)
    print("Precision:", precision)
    print("Recall:   ", recall)
    print("F1:       ", f1)
    print("MCC:      ", mcc)


QQQ:
Mean:      0.5765407554671969
Accuracy:  0.5089463220675944
Precision: 0.5682539682539682
Recall:    0.6172413793103448
F1:        0.5917355371900826
MCC:       -0.02170909523771242

BB:
Mean:      0.4691848906560636
Accuracy:  0.5129224652087475
Precision: 0.44155844155844154
Recall:    0.1440677966101695
F1:        0.21725239616613418
MCC:       -0.02353542165237785

M:
Mean:      0.48111332007952284
Accuracy:  0.48111332007952284
Precision: 0.470404984423676
Recall:    0.6239669421487604
F1:        0.5364120781527532
MCC:       -0.028462880302312477

CMG:
Mean:      0.5546719681908548
Accuracy:  0.4532803180914513
Precision: 0.5117647058823529
Recall:    0.3118279569892473
F1:        0.38752783964365256
MCC:       -0.06168435437488571


In [10]:
for ticker_symbol, results in all_results.items():
    totalIn = 0
    totalOut = 0
    totalInvested = 0
    totalStoodOut = 0
    alltradein = 0
    alltradeout = 0
    perftradein = 0
    perftradeout = 0

    price = prices[ticker_symbol]

    for date, pred, score in zip(results["Date"], results["Prediction"], results["Score"]):
        current_close = price.loc[date, "Close"]

        next_idx = price.index.get_loc(date) + 1
        if next_idx >= len(price):
            continue

        next_close = price.iloc[next_idx]["Close"]
        ret_percent = ((next_close - current_close) / current_close) * 100

        if pred == 1:
            totalIn += 100
            totalInvested += 1
            totalOut += 100 + ret_percent
        else:
            totalStoodOut += 1

        alltradein += 100
        alltradeout += 100 + ret_percent

        if score == 1:
            perftradein += 100
            perftradeout += 100 + ret_percent

    print(f"\n{ticker_symbol}:")
    print(f"  Model:    Invested ${totalIn:.0f}, Returned ${totalOut:.2f}, Profit ${totalOut - totalIn:.2f} ({totalInvested} trades, {totalStoodOut} skipped)")
    print(f"  Buy&Hold: Invested ${alltradein:.0f}, Returned ${alltradeout:.2f}, Profit ${alltradeout - alltradein:.2f}")
    
    print(f"  Perfect:  Invested ${perftradein:.0f}, Returned ${perftradeout:.2f}, Profit ${perftradeout - perftradein:.2f}")


QQQ:
  Model:    Invested $31400, Returned $31443.24, Profit $43.24 (314 trades, 188 skipped)
  Buy&Hold: Invested $50200, Returned $50243.01, Profit $43.01
  Perfect:  Invested $29000, Returned $29296.20, Profit $296.20

BB:
  Model:    Invested $7700, Returned $7652.53, Profit $-47.47 (77 trades, 425 skipped)
  Buy&Hold: Invested $50200, Returned $50150.43, Profit $-49.57
  Perfect:  Invested $23600, Returned $24110.42, Profit $510.42

M:
  Model:    Invested $32000, Returned $31929.13, Profit $-70.87 (320 trades, 182 skipped)
  Buy&Hold: Invested $50200, Returned $50095.15, Profit $-104.85
  Perfect:  Invested $24200, Returned $24783.07, Profit $583.07

CMG:
  Model:    Invested $17000, Returned $17022.70, Profit $22.70 (170 trades, 332 skipped)
  Buy&Hold: Invested $50200, Returned $50295.25, Profit $95.25
  Perfect:  Invested $27900, Returned $28349.91, Profit $449.91
